
## Phase 3: Exploratory Data Analysis (EDA) and Clinical Hypotheses

### Analytical Objectives

In this phase, Exploratory Data Analysis (EDA) is carried out to better understand the hospital readmission dataset and identify patterns that may be associated with patients returning to the hospital within 30 days. Using the cleaned dataset enriched with ICD-9 diagnostic descriptions, the analysis focuses on four main areas.

1. **Readmission Distribution:** Examine the distribution of the `readmitted` variable, which contains three categories: `NO`, `>30`, and `<30`. Particular attention is given to patients readmitted within 30 days, as this group is relevant to the CMS Hospital Readmissions Reduction Program (HRRP). The analysis also considers whether an imbalance exists between the three categories.

2. **Patient Demographics and Healthcare Differences:** Explore patient encounter patterns across 10-year age groups. Readmission rates are also compared across race and gender categories to identify possible demographic differences in hospital readmission patterns.

3. **Medication Patterns and Diabetes Management:** Investigate readmission outcomes among different diabetes medication groups, including insulin monotherapy, combination therapy, oral medication only, and patients who received no diabetes medication. The analysis also examines whether changes in medication dosage (`change == 'Ch'`) are associated with differences in readmission outcomes.

4. **Hospital Stay and Discharge Analysis:** Examine relationships between numerical clinical variables to identify potential multicollinearity. Investigate whether the length of hospital stay (`time_in_hospital`) is related to the number of laboratory procedures (`num_lab_procedures`). Finally, compare readmission patterns among patients discharged to their homes and those transferred to Skilled Nursing Facilities (SNFs).


### 3.1 The Readmission Landscape: Target Variable and Class Imbalance

#### Understanding the Readmission Data

Before investigating the factors that may influence hospital readmissions, it is important to understand how readmissions are distributed across the dataset. The `readmitted` variable contains three possible outcomes: `NO`, `>30`, and `<30`, representing patients who were not readmitted, readmitted after 30 days, and readmitted within 30 days, respectively.

The analysis gives particular attention to the `<30` category because the CMS Hospital Readmissions Reduction Program (HRRP) focuses on certain unplanned hospital readmissions occurring within 30 days of discharge. Hospitals may face financial penalties when their risk-adjusted readmission performance does not meet the programme's requirements.

Patients readmitted within 30 days account for approximately 11.3% of all encounters, making this the smallest category in the dataset. This imbalance is important because comparing raw encounter counts alone may hide differences in readmission patterns among patient groups.

To make these comparisons more meaningful, a new binary variable called `readmitted_30d` is created. It identifies whether an encounter resulted in a readmission within 30 days. Using this variable, readmission rates can be calculated as percentages rather than relying only on total counts. This provides a clearer basis for comparing readmission patterns across different groups.

In [ ]:
# Engineer explicit binary 30-day readmission indicator for rate modeling
df_cleaned['readmitted_30d'] = (df_cleaned['readmitted'] == '<30').astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
order = ['NO', '>30', '<30']
counts = df_cleaned['readmitted'].value_counts()[order]

# Left: Donut Chart of Proportions
wedges, texts, autotexts = ax1.pie(
    counts, labels=['Not Readmitted (NO)', 'Readmitted >30 Days', 'Readmitted <30 Days (Penalty)'],
    autopct='%1.1f%%', startangle=140, colors=colors,
    explode=(0, 0, 0.08), pctdistance=0.78,
    wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2)
)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)
ax1.set_title("Proportional Breakdown of Encounters", fontweight='bold', fontsize=13)

# Right: Horizontal Bar Chart of Exact Volumes
y_pos = np.arange(len(order))
bars = ax2.barh(y_pos, counts.values, color=colors, height=0.55, edgecolor='black', alpha=0.9)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(['NO (Stable)', '>30 Days (Late)', '<30 Days (HRRP Target)'], fontweight='bold')
ax2.invert_yaxis()
ax2.set_xlabel("Number of Hospital Encounters", fontsize=11)
ax2.set_title("Encounter Volume & Class Imbalance", fontweight='bold', fontsize=13)

for bar in bars:
    width = bar.get_width()
    ax2.annotate(f" {width:,} ({width/len(df_cleaned)*100:.2f}%)",
                 xy=(width, bar.get_y() + bar.get_height() / 2),
                 va='center', ha='left', fontweight='bold', fontsize=10)
ax2.set_xlim(0, counts.max() * 1.28)

plt.tight_layout()
plt.show()

### 3.2 Demographic Profiling: Age Distribution and Readmission Patterns

#### 3.2 Age as a Factor in Patient Outcomes

Age is worth examining when studying hospital readmissions among patients with Type 2 diabetes. As patients get older, they may become more vulnerable to diabetes-related complications, including macrovascular and microvascular diseases. Such complications can increase the need for hospital treatment and make recovery after discharge more challenging.

For this analysis, patient encounters are divided into 10-year age groups. The total number of encounters and the corresponding 30-day readmission rate are then compared using a dual-axis chart. The primary axis represents the number of hospital encounters, while the secondary axis displays the readmission rate as a percentage.

Looking at both measurements together helps reveal whether certain age groups have more hospital encounters and whether those groups also experience higher rates of readmission. It also helps identify age categories that may need additional attention when considering patient care and discharge planning.

In [ ]:
# Step 3.2: Dual-Axis Analysis of Age Distribution vs. 30-Day Readmission Rate
age_stats = df_cleaned.groupby('age').agg(
    Encounters=('encounter_id', 'count'),
    Readmission_Rate=('readmitted_30d', lambda x: x.mean() * 100)
).reindex(sorted(df_cleaned['age'].unique())).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 5.5))

# Bar chart: Encounter Volume
bars = ax1.bar(age_stats['age'], age_stats['Encounters'], color='#3498db', alpha=0.75, width=0.6, label='Encounter Volume')
ax1.set_xlabel("Patient Age Bracket (10-Year Intervals)", fontweight='bold', fontsize=11)
ax1.set_ylabel("Number of Encounters", color='#2980b9', fontweight='bold', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#2980b9')
ax1.grid(False)

for bar in bars:
    h = bar.get_height()
    ax1.annotate(f"{h:,}", (bar.get_x() + bar.get_width()/2., h + 300),
                 ha='center', va='bottom', fontsize=8.5, color='#2c3e50')

# Line chart: 30-Day Readmission Rate (%)
ax2 = ax1.twinx()
ax2.plot(age_stats['age'], age_stats['Readmission_Rate'], color='#c0392b', marker='o', linewidth=2.5, markersize=8, label='30-Day Readmission Rate (%)')
ax2.set_ylabel("30-Day Readmission Rate (%)", color='#c0392b', fontweight='bold', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#c0392b')
ax2.axhline(df_cleaned['readmitted_30d'].mean() * 100, color='#7f8c8d', linestyle='--', linewidth=1.2, label='Network Avg Rate')
ax2.set_ylim(0, max(age_stats['Readmission_Rate']) * 1.25)

for i, txt in enumerate(age_stats['Readmission_Rate']):
    ax2.annotate(f"{txt:.1f}%", (age_stats['age'][i], age_stats['Readmission_Rate'][i] + 0.5),
                 ha='center', fontweight='bold', color='#c0392b', fontsize=9)

plt.title("Inpatient Cohort Volume vs. 30-Day Readmission Risk Across Age Brackets", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 3.2 Intersectional Equity Analysis: Readmission by Race and Biological Sex

#### Examining Differences Across Demographic Groups

Patient readmission patterns may differ across demographic groups, making it important to examine whether all groups experience similar outcomes. Race and biological sex are considered together in this analysis to explore whether differences become more noticeable when these two characteristics are examined at the same time.

A two-dimensional heatmap is used to display 30-day readmission rates across the different race and biological sex categories. In addition, a dumbbell plot is used to compare the readmission rates of each demographic group against the overall average of the VHN hospital network.

These visual comparisons help identify groups with relatively higher or lower readmission rates and highlight possible differences in patient outcomes. The findings can provide a starting point for further investigation into healthcare access, patient characteristics, and other factors that may contribute to these differences.

In [ ]:
# Step 3.2: Intersectional 30-Day Readmission Rates by Race and Biological Sex
df_demo = df_cleaned[df_cleaned['race'].notna() & df_cleaned['gender'].isin(['Male', 'Female'])].copy()
pivot_demo = df_demo.pivot_table(index='race', columns='gender', values='readmitted_30d', aggfunc=lambda x: x.mean() * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Annotated Heatmap Matrix
sns.heatmap(pivot_demo, annot=True, fmt=".2f", cmap="YlOrRd", cbar_kws={'label': '30-Day Readmission (%)'}, ax=ax1, linewidths=1)
ax1.set_title("Intersectional Risk Matrix (Race x Sex)", fontweight='bold', fontsize=13)
ax1.set_xlabel("Biological Sex", fontweight='bold')
ax1.set_ylabel("Reported Race", fontweight='bold')

# Right: Dumbbell / Dot Plot Comparison
races = pivot_demo.index
y_pos = np.arange(len(races))
ax2.hlines(y=y_pos, xmin=pivot_demo['Male'], xmax=pivot_demo['Female'], color='grey', alpha=0.6, linewidth=2.5)
ax2.scatter(pivot_demo['Male'], y_pos, color='#2980b9', s=100, label='Male', zorder=3)
ax2.scatter(pivot_demo['Female'], y_pos, color='#e74c3c', s=100, label='Female', zorder=3)

ax2.axvline(df_cleaned['readmitted_30d'].mean() * 100, color='black', linestyle='--', alpha=0.6, label='Network Average')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(races, fontweight='bold')
ax2.set_xlabel("30-Day Readmission Rate (%)", fontweight='bold')
ax2.set_title("Gender Disparity Gap Within Each Racial Group", fontweight='bold', fontsize=13)
ax2.legend()

plt.tight_layout()
plt.show()

### 3.3 Medication Analysis: Insulin vs Antidiabetics

#### 3.3 Classifying Diabetes Treatment Approaches

The dataset contains information about insulin treatment and 22 additional oral antidiabetic medications, including `metformin`, `glipizide`, and `pioglitazone`. Examining these medications individually would make it difficult to compare overall treatment patterns. Therefore, patient encounters are divided into four medication groups based on the treatments recorded in the dataset.

1. **Insulin Monotherapy:** Encounters where insulin is recorded as the only diabetes medication, with no oral antidiabetic agents.

2. **Combination Therapy (Insulin + Oral):** Encounters where insulin is used together with at least one oral antidiabetic medication.

3. **Oral Agents Only:** Encounters where patients receive one or more oral antidiabetic medications without insulin.

4. **No Diabetes Medication:** Encounters where no diabetes medication is recorded during the hospital stay.

After assigning encounters to these groups, their 30-day readmission rates are compared to identify differences in outcomes between treatment categories.

One of the main questions explored in this analysis is whether patients receiving insulin, either alone or alongside oral medications, have higher readmission rates than those treated exclusively with oral agents. This may help reveal associations between treatment patterns and patient outcomes, although medication use alone cannot establish the severity of a patient's condition or prove that a particular treatment causes readmission.

In [ ]:
# Step 3.3: Medication Regimen Classification & Readmission Efficacy
oral_agents = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

def classify_medication_regimen(row):
    has_insulin = (row['insulin'] != 'No')
    has_oral = any(row[med] != 'No' for med in oral_agents)
    if has_insulin and has_oral: return 'Combination (Insulin + Oral)'
    elif has_insulin and not has_oral: return 'Insulin Monotherapy'
    elif not has_insulin and has_oral: return 'Oral Agents Only'
    else: return 'No Diabetes Medication'

df_cleaned['Medication_Cohort'] = df_cleaned.apply(classify_medication_regimen, axis=1)

med_summary = df_cleaned.groupby('Medication_Cohort')['readmitted_30d'].mean().mul(100).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Left: Horizontal Lollipop Chart
y_pos = np.arange(len(med_summary))
ax1.hlines(y=y_pos, xmin=0, xmax=med_summary.values, color='#16a085', alpha=0.8, linewidth=3)
ax1.plot(med_summary.values, y_pos, "o", markersize=11, color='#e74c3c')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(med_summary.index, fontweight='bold')
ax1.axvline(df_cleaned['readmitted_30d'].mean() * 100, color='black', linestyle='--', label='Network Avg')
ax1.set_xlim(0, 16)
ax1.set_xlabel("30-Day Readmission Rate (%)", fontweight='bold')
ax1.set_title("30-Day Risk Ranking by Pharmacological Regimen", fontweight='bold', fontsize=12)
for i, v in enumerate(med_summary.values):
    ax1.text(v + 0.4, i, f"{v:.2f}%", va='center', fontweight='bold', color='#c0392b')
ax1.legend(loc='lower right')

# Right: 100% Stacked Bar Chart across all readmission states
ct_med = pd.crosstab(df_cleaned['Medication_Cohort'], df_cleaned['readmitted'], normalize='index')[['NO', '>30', '<30']] * 100
ct_med.loc[med_summary.index].plot(kind='barh', stacked=True, color=['#2ecc71', '#f39c12', '#e74c3c'], ax=ax2, edgecolor='white')
ax2.set_title("Full Readmission Outcome Split by Regimen (%)", fontweight='bold', fontsize=12)
ax2.set_xlabel("Percentage of Cohort (%)", fontweight='bold')
ax2.set_ylabel("")
ax2.legend(title='Readmitted', loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=3)

plt.tight_layout()
plt.show()